# Generate Scores

## Notebook Preferences

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format="retina"

## Libraries

In [ ]:
import buckaroo  # noqa: F401
import lamindb as ln
import spatialdata as sd
import pandas as pd
import matplotlib.pyplot as plt
import nbl
import anndata as ad
import scanpy as sc
import seaborn.objects as so
import itertools
from matplotlib.figure import Figure
from upath import UPath

In [ ]:
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("future.infer_string", False)
pd.set_option("mode.copy_on_write", False)

In [ ]:
from seaborn import axes_style

theme_dict = {**axes_style("whitegrid"), "grid.linestyle": ":", "axes.facecolor": "w", "axes.edgecolor": "slategray"}
so.Plot.config.theme.update(theme_dict)

In [ ]:
# ln.context.uid = "c9lSn4d8xU4f0000"
# ln.context.version = "1"
ln.settings.sync_git_repo = "https://github.com/karadavis-lab/nbl.git"
ln.track(uid="a5bTVIqiPhn00000")

In [ ]:
ln.setup.settings.instance

## Load NBL Sdata

In [ ]:
nbl_sdata = sd.read_zarr(ln.Artifact.filter(key__contains="nbl.zarr", is_latest=True).one().path)

In [ ]:
nbl_wc = nbl_sdata.tables["arcsinh_shift_0_scale_150"]

nbl_sdata.tables["nbl_wc"] = nbl_wc[nbl_wc.obs["pixie_cluster"] == "NBL_cell"].copy()
nbl_sdata.tables["nbl_wc"].uns = nbl_sdata.tables["whole_cell"].uns

nbl_sdata.update_annotated_regions_metadata(table=nbl_sdata.tables["nbl_wc"])

In [ ]:
nbl.util.write_elements(sdata=nbl_sdata, elements={"tables": "nbl_wc"})

In [ ]:
nbl_wc = nbl_sdata.tables["nbl_wc"]

### Get Marker Groups

In [ ]:
adrenergic_markers = nbl.ln.cell_marker_set_catalog("adrenergic", return_type="names")

In [ ]:
mesenchymal_markers = nbl.ln.cell_marker_set_catalog("mesenchymal", return_type="names")

In [ ]:
neuroblastoma_markers = nbl.ln.cell_marker_set_catalog("neuroblastoma", "names")

Normalize by Area of each cell

In [ ]:
nbl.pp.normalize_by_area(sdata=nbl_sdata, table_names="nbl_wc", method="layer", write=False, inplace=True)

## Compute Scores

In [ ]:
marker_groups = {
    "ADRN": adrenergic_markers,
    "ADRN_no_TH": adrenergic_markers[:2],
    "MESN": mesenchymal_markers,
}
nbl.tl.compute_marker_means(
    nbl_sdata, table_name="nbl_wc", layer="area_normalized", marker_groups=marker_groups, inplace=True
)

In [ ]:
for score_method, obs_1 in itertools.product(nbl.tl._scores.keys(), ["ADRN", "ADRN_no_TH"]):
    nbl.tl.compute_score(
        sdata=nbl_sdata,
        table_name="nbl_wc",
        obs_1=f"{obs_1}_mean",
        obs_2="MESN_mean",
        score_method=score_method,
        score_col_name=f"{score_method}_no_TH" if "no_TH" in obs_1 else f"{score_method}",
        eps=1e-4,
    )

## Clustering with Area Normalized Arcsinh Transformed Markers

In [ ]:
nbl.pp.neighbors(
    sdata=nbl_sdata,
    table_name="nbl_wc",
    layer="area_normalized",
    key_added="area_norm_neighbors",
    vars=[*marker_groups["ADRN_no_TH"], *marker_groups["MESN"]],
    inplace=True,
    method="umap",
    metric="euclidean",
)

In [ ]:
q = nbl.tl.diffmap(
    sdata=nbl_sdata,
    table_name="nbl_wc",
    neighbors_key="area_norm_neighbors",
    vars=[*marker_groups["ADRN_no_TH"], *marker_groups["MESN"]],
    layer="area_normalized",
    inplace=False,
)

In [ ]:
nbl.tl.leiden(
    sdata=nbl_sdata,
    table_name="nbl_wc",
    neighbors_key="area_norm_neighbors",
    layer="area_normalized",
    vars=[*marker_groups["ADRN_no_TH"], *marker_groups["MESN"]],
    key_added="area_norm_leiden",
    flavor="igraph",
    n_iterations=-1,
)

Subset on just the Diagnosis Classification samples.

In [ ]:
nbl_wc_diagnosis: ad.AnnData = nbl_sdata.tables["nbl_wc"][
    nbl_sdata.tables["nbl_wc"].obs["Classification"] == "Diagnosis", :
].copy()

In [ ]:
nbl_sdata.tables["nbl_wc_diagnosis"] = nbl_sdata.update_annotated_regions_metadata(table=nbl_wc_diagnosis)


nbl.util.write_elements(sdata=nbl_sdata, elements={"tables": "nbl_wc_diagnosis"})

In [ ]:
var_names = {
    "ADRN_no_TH": adrenergic_markers[:2],
    "MESN": mesenchymal_markers,
}

In [ ]:
figures_upath = UPath("../../../data/db/figures/scoring")
figures_upath.mkdir(exist_ok=True, parents=True)

In [ ]:
fig = plt.figure(figsize=(6, 16), dpi=300)
ax = fig.subplots(nrows=1, ncols=1)
sc.pl.dotplot(
    adata=nbl_sdata.tables["nbl_wc"],
    var_names=var_names,
    groupby=["area_norm_leiden"],
    layer="area_normalized",
    log=True,
    return_fig=True,
    cmap="viridis",
    title="Arcsinh Transformed, Area Normalized Leiden Clusters (without TH)",
    ax=ax,
).add_totals(show=True, color="xkcd:ocean blue").legend(show=True).make_figure()

fig_path: UPath = figures_upath / "leiden_clusters_area_norm_without_TH_dotplot.pdf"
fig.savefig(fig_path)
artifact = ln.Artifact(
    data=fig_path,
    description="Leiden Clusters Area Normalized without TH Dotplot",
)
artifact.save()

In [ ]:
nbl_wc_diagnosis.obs.columns

In [ ]:
# score = "log_ratio_no_TH"
for score in ["ratio_no_TH", "log_ratio_no_TH", "normalized_difference_no_TH", "scaled_difference_no_TH"]:
    for risk in ["High", "Intermediate", "Low"]:
        fig: Figure = plt.figure(figsize=(16, 10), layout="constrained")
        fig.suptitle(t=f"{score} Scores for Diagnosis Samples | Risk: {risk}")

        score_subfig, marker_subfig = fig.subfigures(nrows=1, ncols=2)

        markers_subfig, score_v_marker_subfig = marker_subfig.subfigures(nrows=2, ncols=1)

        score_ax = score_subfig.subplots(nrows=1, ncols=1)

        marker_axes = markers_subfig.subplots(nrows=2, ncols=3, sharex=True, sharey=True)
        svm_subfigures = score_v_marker_subfig.subfigures(nrows=1, ncols=2)
        nbl_wc_risk: ad.AnnData = nbl_wc_diagnosis[nbl_wc_diagnosis.obs["Risk"] == risk, :]

        # Plotting histogram on ax1
        plot1 = (
            so.Plot(nbl_wc_risk.obs, x=score)
            .add(so.Bars(), so.Hist())
            .scale(x="symlog")
            .label(
                x=r"$ \ln{\left(\frac{ \bar{\bf{M}} + \epsilon} { \bar{\bf{A}} + \epsilon}\right)}$",
                y="Count",
                title="Scores",
            )
        )
        plot1.on(score_ax).plot()

        # Create a secondary y-axis
        ax2 = score_ax.twinx()

        # Plotting cumulative KDE on ax2
        plot2 = so.Plot(nbl_wc_risk.obs, x=score).add(so.Line(color="xkcd:cerulean"), so.KDE(cumulative=True))

        plot2.on(ax2).plot()

        # Set labels for ax2
        ax2.set_ylabel("CDF")
        ax2.yaxis.tick_right()

        for ax, marker in zip(marker_axes.flat, [*mesenchymal_markers, *adrenergic_markers], strict=False):
            so.Plot(data=nbl_wc_risk[:, [marker]].to_df(), x=marker).add(so.Bars(), so.Hist()).on(ax).plot()

        for sf, marker_group in zip(svm_subfigures.flat, [mesenchymal_markers, adrenergic_markers], strict=False):
            a = nbl_wc_risk.to_df(layer="area_normalized").copy()
            b = nbl_wc_risk.obs[score].copy()
            so.Plot(data=a.join(b), x=score).pair(y=marker_group).add(so.Dots(marker=".", pointsize=0.5)).on(sf).label(
                x=score
            ).plot()

        markers_subfig.suptitle(t="Marker Distribution (arcsinh transformed)")
        score_v_marker_subfig.suptitle(t="Score vs. Marker Area Normalized Arcsinh Transformed Value")
        fig_path: UPath = figures_upath / f"{score}_scores_{risk}.pdf"
        fig.savefig(fig_path)
        artifact = ln.Artifact(
            data=fig_path,
            description=f"{score} Scores for Diagnosis Samples -- Risk: {risk}",
        )
        artifact.save()

        fig.show(warn=False)

In [ ]:
for score in ["ratio_no_TH", "log_ratio_no_TH", "normalized_difference_no_TH", "scaled_difference_no_TH"]:
    fig = plt.figure(figsize=(16, 10), dpi=300)
    subfigs = fig.subfigures(nrows=1, ncols=2, wspace=0.15)

    score_ax = subfigs[0].subplots(nrows=1, ncols=1, sharex=True, sharey=True)
    dist_ax = subfigs[1].subplots(nrows=2, ncols=3, sharex=True, sharey=True)

    b = (
        nbl_wc_diagnosis[:, [*mesenchymal_markers, *adrenergic_markers]]
        .to_df()
        .merge(right=nbl_wc_diagnosis.obs["Risk"], left_index=True, right_index=True)
    )

    # Plotting histogram on ax1
    plot1 = (
        so.Plot(nbl_wc_diagnosis.obs, x=score, color="Risk")
        .add(so.Bars(), so.Hist())
        .scale(x="symlog")
        .label(
            x=score,
            y="Count",
            title="Diagnosis -- Scores",
        )
    )
    plot1.on(score_ax).plot()

    # Create a secondary y-axis
    score_ax2 = score_ax.twinx()

    # Plotting cumulative KDE on ax2
    plot2 = (
        so.Plot(nbl_wc_diagnosis.obs, x=score, color="Risk").add(so.Line(), so.KDE(cumulative=True)).scale(x="symlog")
    )

    plot2.on(score_ax2).plot()

    # Set labels for ax2
    score_ax2.set_ylabel("CDF")
    score_ax2.yaxis.tick_right()

    for ax, marker in zip(dist_ax.flat, [*mesenchymal_markers, *adrenergic_markers], strict=False):
        so.Plot(data=b, x=marker, color="Risk").add(so.Bars(), so.Hist()).on(ax).label(x=marker, y="Count").plot()
    subfigs[1].suptitle(t="Marker Distributions")
    subfigs[1].subplots_adjust()
    fig_path: UPath = figures_upath / f"diagnosis_scores_w_marker_dists_{score}.pdf"

    fig.savefig(fig_path)

In [ ]:
q = nbl.tl.umap(
    sdata=nbl_sdata,
    table_name="nbl_wc_diagnosis",
    layer="area_normalized",
    neighbors_key="area_norm_neighbors",
    vars=[*adrenergic_markers[:2], *mesenchymal_markers],
    gamma=2,
)

In [ ]:
sc.pl.umap(adata=q, color="normalized_difference_no_TH")

In [ ]:
nbl.util.write_elements(sdata=nbl_sdata, elements={"tables": ["nbl_wc_diagnosis"]})

In [ ]:
var_names = {
    "Adrenergic": adrenergic_markers[:2],
    "Mesenchymal": mesenchymal_markers,
}

## Quantiles

In [ ]:
nbl.tl.quantile(
    sdata=nbl_sdata,
    table_name="nbl_wc_diagnosis",
    layer="area_normalized",
    var="CD45",
    q=[0.99, 0.95, 0.90],
    inplace=False,
    write=False,
)

In [ ]:
# for q in [0.99, 0.95, 0.90]:
#     nbl_diagnosis_wc_q: ad.AnnData = nbl.tl.filter_obs_names_by_quantile(
#         sdata=nbl_sdata, table_name="nbl_wc_diagnosis", var="CD45", q=q, method="upper", layer="area_normalized"
#     )
#     for risk in ["High", "Intermediate", "Low"]:
#         fig: Figure = plt.figure(figsize=(16, 10), layout="constrained")
#         fig.suptitle(t=f"Log Ratio Scores for Diagnosis | Risk: {risk} | Quantile: {q}")

#         score_subfig, marker_subfig = fig.subfigures(nrows=1, ncols=2)

#         markers_subfig, score_v_marker_subfig = marker_subfig.subfigures(nrows=2, ncols=1)

#         score_ax = score_subfig.subplots(nrows=1, ncols=1)

#         marker_axes = markers_subfig.subplots(nrows=2, ncols=3, sharex=True, sharey=True)
#         svm_subfigures = score_v_marker_subfig.subfigures(nrows=1, ncols=2)
#         nbl_wc_risk: ad.AnnData = nbl_diagnosis_wc_q[nbl_diagnosis_wc_q.obs["Risk"] == risk, :]

#         # Plotting histogram on ax1
#         plot1 = (
#             so.Plot(nbl_wc_risk.obs, x="log_ratio_no_TH")
#             .add(so.Bars(), so.Hist())
#             .scale(x="symlog")
#             .label(
#                 x=r"$ \ln{\left(\frac{ \bar{\bf{M}} + \epsilon} { \bar{\bf{A}} + \epsilon}\right)}$",
#                 y="Count",
#                 title="Scores",
#             )
#         )
#         plot1.on(score_ax).plot()

#         # Create a secondary y-axis
#         ax2 = score_ax.twinx()

#         # Plotting cumulative KDE on ax2
#         plot2 = (
#             so.Plot(nbl_wc_risk.obs, x="log_ratio_no_TH")
#             .add(so.Line(color="xkcd:cerulean"), so.KDE(cumulative=True))
#             .scale(x="symlog")
#         )

#         plot2.on(ax2).plot()

#         # Set labels for ax2
#         ax2.set_ylabel("Cumulative Probability")
#         ax2.yaxis.tick_right()

#         for ax, marker in zip(marker_axes.flat, [*mesenchymal_markers, *adrenergic_markers], strict=False):
#             so.Plot(data=nbl_wc_risk[:, [marker]].to_df(), x=marker).add(so.Bars(), so.Hist()).on(ax).plot()

#         for sf, marker_group in zip(svm_subfigures.flat, [mesenchymal_markers, adrenergic_markers], strict=False):
#             a = nbl_wc_risk.to_df(layer="area_normalized").copy()
#             b = nbl_wc_risk.obs["log_ratio_no_TH"].copy()
#             so.Plot(data=a.join(b), x="log_ratio_no_TH").pair(y=marker_group).add(so.Dots(marker=".", pointsize=5)).on(
#                 sf
#             ).plot()
#         markers_subfig.suptitle(t="Marker Distribution (arcsinh transformed)")
#         score_v_marker_subfig.suptitle(t="Score vs. Marker")
#         fig_path: UPath = figures_upath / f"quantile_{q}_scores_{risk}.pdf"
#         fig.savefig(fig_path)
#         artifact = ln.Artifact(
#             data=fig_path,
#             description=f"Quantile {q} Scores for Diagnosis Samples -- Risk: {risk}",
#         )
#         artifact.save()
#         fig.show(warn=False)

In [ ]:
# metric = "normalized_difference_no_TH"
# vmax: float = nbl_sdata.tables["nbl_wc_diagnosis"].obs[metric].max()

# vmin: float = nbl_sdata.tables["nbl_wc_diagnosis"].obs[metric].min()

In [ ]:
# nbl_sdata.tables["nbl_wc_diagnosis"].obs.columns

In [ ]:
# # fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(16, 16), dpi=300, layout="constrained", sharex=True, sharey=True)

# # for fov, ax in zip(fovs_for_fig, axes.flat, strict=False):
# nbl_sdata.filter_by_coordinate_system(fov).pl.render_labels(
#     element=f"{fov}_whole_cell",
#     color="pixie_cluster",
#     # groups=a,
#     table_name="whole_cell",
#     outline_alpha=1,
#     method="datashader",
#     fill_alpha=0.99,
#     scale="full",
# ).pl.render_labels(
#     element=f"{fov}_whole_cell",
#     color=metric,
#     outline_alpha=1,
#     table_name="nbl_wc_diagnosis",
#     method="datashader",
#     scale="full",
#     cmap="viridis",
#     fill_alpha=0.99,
#     norm=colors.Normalize(vmin=vmin, vmax=vmax),
# ).pl.show()

# # nbl.util.remove_ticks(f=ax, axis="xy")
# # fig.savefig("many_fovs.png", dpi=300)

In [ ]:
# nbl_sdata.filter_by_coordinate_system(fov).pl.render_labels(
#     element=f"{fov}_whole_cell",
#     color="pixie_cluster",
#     table_name="whole_cell",
#     outline_alpha=1,
#     method="datashader",
#     fill_alpha=0.99,
#     scale="full",
# ).pl.show()

In [ ]:
ln.finish()